# EnSIMem: tiny entity-structured memory demo

This notebook implements a small, deterministic approximation of the central idea in **EnSIMem: Entity-Structured Indexing for Long-Term Agent Memory**.

The index is an **address system**, not the evidence itself: compact entity–property records locate relevant episodes, then retrieval restores the original dialogue turns for grounded reasoning.

Paper: https://arxiv.org/abs/2609.27279  
Official code: https://github.com/RamonMeng/EnSIMem

This toy version uses hand-authored episodes, records, and query requirements. It does not reproduce the paper’s LLM extraction, dense fallback, multimodal handling, or answer-generation experiments.

In [ ]:
from datetime import datetime
from pprint import pprint

# Original dialogue is retained as evidence inside theme-coherent episodes.
episodes = {
    "E1": {
        "theme": "Maya's Chicago trip",
        "timestamp": "2026-03-04T09:00:00",
        "turns": [
            {"turn_id": "T1", "speaker": "user", "text": "Maya traveled to Chicago by helicopter for the conference."},
            {"turn_id": "T2", "speaker": "assistant", "text": "That sounds like a memorable trip."}
        ]
    },
    "E2": {
        "theme": "Maya's earlier drink preference",
        "timestamp": "2026-04-12T15:30:00",
        "turns": [
            {"turn_id": "T3", "speaker": "user", "text": "Maya usually preferred tea in the afternoon."}
        ]
    },
    "E3": {
        "theme": "Maya's updated drink preference",
        "timestamp": "2026-07-20T08:10:00",
        "turns": [
            {"turn_id": "T4", "speaker": "user", "text": "Maya now prefers coffee instead of tea."}
        ]
    },
    "E4": {
        "theme": "Melanie's children's interests",
        "timestamp": "2026-08-03T18:00:00",
        "turns": [
            {"turn_id": "T5", "speaker": "user", "text": "Melanie said one of her children loves nature walks."}
        ]
    },
    "E5": {
        "theme": "Melanie's children's interests",
        "timestamp": "2026-08-17T12:45:00",
        "turns": [
            {"turn_id": "T6", "speaker": "user", "text": "Her other child is fascinated by dinosaurs."}
        ]
    },
}
print(f"Stored {len(episodes)} episodes with original dialogue.")

In [ ]:
# Retrieval-facing records: [entity][entity_type][property:value] -> provenance.
index = [
    {"entity": "Maya", "entity_type": "person", "property": "destination", "value": "Chicago", "episode_id": "E1", "source_turn_ids": ["T1"]},
    {"entity": "Maya", "entity_type": "person", "property": "means_of_transport", "value": "helicopter", "episode_id": "E1", "source_turn_ids": ["T1"]},
    {"entity": "Maya", "entity_type": "person", "property": "drink_preference", "value": "tea", "episode_id": "E2", "source_turn_ids": ["T3"]},
    {"entity": "Maya", "entity_type": "person", "property": "drink_preference", "value": "coffee", "episode_id": "E3", "source_turn_ids": ["T4"]},
    {"entity": "Melanie's children", "entity_type": "group", "property": "likes", "value": "nature walks", "episode_id": "E4", "source_turn_ids": ["T5"]},
    {"entity": "Melanie's children", "entity_type": "group", "property": "likes", "value": "dinosaurs", "episode_id": "E5", "source_turn_ids": ["T6"]},
]
pprint(index[:2])

In [ ]:
def _same(left, right):
    return left.casefold().replace(" ", "_") == right.casefold().replace(" ", "_")

def retrieve(requirements, query_type="point"):
    """Structured lookup followed by query-type-aware coverage and evidence expansion."""
    matches = []
    for requirement in requirements:
        for record in index:
            compatible = (
                _same(record["entity"], requirement["entity"])
                and _same(record["property"], requirement["property"])
            )
            if requirement.get("value") is not None:
                compatible = compatible and _same(record["value"], requirement["value"])
            if compatible:
                matches.append(record)

    # Deduplicate records and order them by their source episode time.
    unique = {(r["entity"], r["property"], r["value"], r["episode_id"]): r for r in matches}
    matches = sorted(unique.values(), key=lambda r: datetime.fromisoformat(episodes[r["episode_id"]]["timestamp"]))

    # Point queries need one supported value; temporal queries retain the timeline;
    # aggregation and compositional queries retain all matching evidence.
    if query_type == "point" and matches:
        matches = [matches[-1]]
    elif query_type not in {"temporal", "aggregation", "compositional", "point"}:
        raise ValueError(f"Unknown query type: {query_type}")

    evidence_buffer = []
    for record in matches:
        episode = episodes[record["episode_id"]]
        source_ids = set(record["source_turn_ids"])
        evidence_buffer.append({
            "matched_address": f"[{record['entity']}][{record['entity_type']}][{record['property']}:{record['value']}]",
            "episode_id": record["episode_id"],
            "timestamp": episode["timestamp"],
            "source_turns": [turn for turn in episode["turns"] if turn["turn_id"] in source_ids],
        })
    return evidence_buffer

print("Retriever ready.")

In [ ]:
# Point query: one entity-property address is sufficient.
point_requirements = [{"entity": "Maya", "property": "means of transport"}]
point_buffer = retrieve(point_requirements, query_type="point")
print("Question: How did Maya travel to Chicago?")
pprint(point_buffer)

In [ ]:
# Temporal query: preserve all dated evidence so a downstream reasoner can see the update.
temporal_requirements = [{"entity": "Maya", "property": "drink preference"}]
temporal_buffer = retrieve(temporal_requirements, query_type="temporal")
print("Question: How did Maya's drink preference change?")
pprint(temporal_buffer)

In [ ]:
# Aggregation query: do not stop after the first relevant episode.
aggregation_requirements = [{"entity": "Melanie's children", "property": "likes"}]
aggregation_buffer = retrieve(aggregation_requirements, query_type="aggregation")
print("Question: What do Melanie's children like?")
pprint(aggregation_buffer)
assert len(aggregation_buffer) == 2

## What the full method adds

In the paper, language models partition raw interaction histories, extract index records, and turn a natural-language request into evidence requirements. Structured lookup is combined with dense fallback retrieval, and the selected source evidence is passed to an answer model. This notebook isolates the paper’s central architectural choice: **compress the access path, not the evidence**.